# Covariate Data Preprocessing

This notebook builds the final **covariate file** for QTL analysis by combining known covariates, genotype principal components (PCs), and hidden factors.


## Description

Covariates correct for confounders so that detected QTLs reflect true genotype-phenotype associations rather than technical or population artifacts. This notebook assembles three kinds of covariates into one file:

1. **Known covariates** — measured variables from the fixed covariate file (e.g. `msex`, `age_death`, `pmi`, `study`).
2. **Genotype PCs** — the top PCs from the genotype PCA (`3_genotype_pca.ipynb`), which capture population structure.
3. **Hidden factors** — unobserved sources of variation (batch effects, technical noise) inferred from the phenotype residuals.

The two steps are: (1) **merge** the genotype PCs with the known covariates, and (2) **compute residuals** on the merged covariates and run **hidden-factor analysis** (PCA on the residuals, with the number of factors chosen by the Marchenko-Pastur distribution). The result is a single covariate file ready for TensorQTL.


## Input Files

| File | Description |
| --- | --- |
| `input/covariate/protocol_example.covariates.tsv` | Fixed/known covariates, one row per sample: `#id`, `msex`, `age_death`, `pmi`, `study`. |
| `output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds` | Genotype PCA model from `3_genotype_pca.ipynb`. |
| `output/genotype/genotype_pca/...prune.pca.scree.txt` | Scree table from `3_genotype_pca.ipynb`; used to pick how many genotype PCs to keep. |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Phenotype bed (from `1_phenotype_preprocessing.ipynb`); used to compute residuals for hidden-factor analysis. |


## Steps


### **Step 1.** Merge genotype PCs with known covariates

Merge the genotype PCA results with the known covariate file. `--k` sets how many genotype PCs to keep; here it is read from the scree table as the number of PCs that together explain < 80% of variance (15 for this dataset). `--tol-cov 0.4` is the maximum allowed missingness rate for covariates. The output is a merged covariate table (known covariates + genotype PCs).


In [ ]:
sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/covariate \
    --pcaFile output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds \
    --covFile input/covariate/protocol_example.covariates.tsv \
    --tol-cov 0.4 \
    --k `awk '$3 < 0.8' output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.scree.txt | tail -1 | cut -f 1`


### **Step 2.** Compute residuals and run hidden-factor analysis

Using the phenotype and the merged covariates from Step 1, this step first regresses out the known covariates to obtain phenotype **residuals** (`Marchenko_PC_1`), then runs PCA on those residuals and keeps the components that pass the **Marchenko-Pastur** significance threshold as **hidden factors** (`Marchenko_PC_2`). `--mean-impute-missing` fills any missing covariate values with their column mean. The final output combines the known covariates, genotype PCs, and hidden factors — this is the covariate file used in downstream QTL analysis.


In [ ]:
sos run pipeline/covariate_hidden_factor.ipynb Marchenko_PC \
    --cwd output/covariate \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz \
    --covFile output/covariate/protocol_example.covariates.protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.gz \
    --mean-impute-missing


# Covariate data preprocessing

This notebook contains workflow of processing covariate files for TensorQTL. It also computes PCA-derived covariates from genotype and phenotype data.

## Methods overview

This workflow is an application of the covariate related sections from the xQTL project pipeline.

## Data Input
- `phenotype data with bed.gz`.
- PCs from genotypes genereated in the [genotype_pca](https://github.com/cumc/brain-xqtl-analysis/tree/main/analysis/Wang_Columbia/ROSMAP/pqtl/genotype_pca) step.
- Fixed covarate file including information such as sex, age at death, pmi etc

First, we need to read covariates data and meta data. meta data is used for match projid and sampleid. In this case, we want to change the projid in the raw cov data to corresponding sample id. Note that some of the projids don't have corresponding sampleid according to the meta list. But it's okay because in the next few steps we will only keep those ids overlapped with phenotype data. You can adjust df_cov.head() to view more.


## Data Output
- `output/data_preprocessing/covariate_data/` This contains all covariates from Genotype PCs, known covariates, and hidden factors.

### Merge covariates and genotype PCA

First, check how many genotype PC we might want to include,  
this screen file is from the pca output of genotype_pca, showing the variance explained by each principal component. 

Here we see 15 PC that will explain 80% variation in the data. Let's include 15 PC in this case. In practice it is suggested that you discuss with your collaborator and/or PI about the choice of PC given results from the previous PCA.

So in --k parameter, we set it as 15.

About `[merge_genotype_pc]`:    

`Aim`: To merge the results of a genotype Principal Component Analysis (PCA) with other covariate data for subsequent analyses.

`Inputs`:

- pcaFile: This is an RDS file, which is the output of the genotype PCA module.    
- covFile: This is a file containing covariate data.    
- k: The number of principal components to retain, defaulting to 20. In this case, we set as 15.    
- outliersFile: A file listing samples considered as outliers.    
- remove_outliers: A flag indicating whether outliers should be removed from the analysis.    
- tol_cov: If tol_cov = -1, then do nothing about missing rate, otherwise it means the maximum allowed missingness rate in covariates.    
- mean_impute: A flag indicating whether missing values in covariates should be imputed with their mean.

`Output`:    

A file that merges the PCA and covariate data. `.plink_qc.prune.pca.gz`.  

In summary, It first checks for sample overlap between the PCA and covariate data, then handles missing values in covariates, and finally merges the processed data and saves it to an output file. So after this cell, you will obtain a file that merges the PCA and covariate data, which can be used for subsequent analyses. 


### Compute residule on merged covariates and perform hidden factor analysis
This step will compute residual on merged covariates(`Marchenko_PC_1`) and perform hidden factor analysis(`Marchenko_PC_2`)

`Background`:    
Hidden factor analysis aims to identify and quantify unobserved (hidden) factors that influence observed variables. In the context of genomics and transcriptomics, these hidden factors can be various sources of variability, such as batch effects, technical artifacts, or other unmeasured biological factors that can influence gene expression levels.

PCA on Residuals:   
Principal Component Analysis (PCA) is a dimensionality reduction technique that captures the major sources of variability in the data. By performing PCA on the residuals (which are the parts of the data unexplained by known covariates), the workflow aims to capture the variability due to hidden factors.

Marchenko-Pastur Distribution:   
The chooseMarchenkoPastur function is used to determine the number of principal components (PCs) to retain. The Marchenko-Pastur distribution is a mathematical tool used to decide how many PCs are likely representing true biological or technical variability (hidden factors) versus random noise. By comparing the eigenvalues (variances) of the PCs to this distribution, one can decide which PCs are likely representing hidden factors.

Principal Components(the output):   
The resulting principal components represent hidden biological factors that contribute to the molecular phenotype variation but are not explained by the known covariates. These factors are:
- Hidden confounders: Unobserved variables that affect gene expression patterns across samples, such as batch effects, population substructure, or unmeasured environmental factors.
- Systematic variation: Sources of correlated variation across genes that may arise from technical factors or biological processes not captured in the covariate file.
- Quality-controlled factors: Only principal components that pass the Marchenko-Pastur significance threshold are retained, ensuring that noise components are excluded.

Let's look at the workflow step by step:  

`Workflow 1: *_1(computing residual on merged covariates)`   

`Aim`: To compute residuals on merged covariates (The portion of phenotypic data that can not be explained by covariates. The effects of known covariates were removed, and the “Pure” signal in the phenotypic data was retained for subsequent hidden factor analysis, allowing PCA to capture real biological variation rather than technical noise).

`Inputs`:     
- phenoFile: A file containing phenotype data.    
- covFile: the merged pca and cov gz.file in the output of `merge_genotype_pc`.   

`Output`:   
`Mic.log2cpm.mic.rosmap_cov.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.snuc_pseudo_bulk.related.plink_qc.extracted.pca.projected.residual.bed.gz`. A compressed .bed.gz file containing the residuals of the merged covariates.    

`In summary`, it first loads pheno, cov file, extracts overlapping Samples, computes residuals in the subset data(only contains overlapped samples) using a linear model fit. Then, the residuals, along with phenotype IDs, are written to a tab-delimited file, and compressed using bgzip and indexed using tabix.

`Workflow 2: Marchenko_PC_2`:      

 `Aim`: To perform Principal Component Analysis (PCA) on the residuals and determine the number of principal components (hidden factor) to retain based on the Marchenko-Pastur distribution.

`Inputs`:     
- Residuals from the previous workflow.
- Covariate file (covFile).

`Output`:   
`Mic.log2cpm.mic.rosmap_cov.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.snuc_pseudo_bulk.related.plink_qc.extracted.pca.projected.Marchenko_PC.gz`. A compressed file containing the principal components(hidden factors--written as Hidden_Factor_PC1, Hidden_Factor_PC2……) and covariates(msex,age_death,pmi ). So this is the final cov file we will actually use in the subsequent analysis(eg: tensorqtl).

`In summary`, the Marchenko_PC_2 workflow is essentially performing hidden factor analysis on the residuals. By extracting the major sources of variability from the residuals using PCA and determining the number of PCs to retain based on the Marchenko-Pastur distribution, the workflow identifies the hidden factors in the data. These hidden factors, represented by the principal components, provide insights into the underlying structure of the data and can be crucial for correcting batch effects, reducing technical noise, and improving the interpretability of downstream analyses.

### Summary analysis of covariates preprocessing results

##### The final covariates list     
The number of covariates listed is 53: msex, age_death, pmi from the raw cov data. PC1-15 from geno_pca. Hidden_Factor_PC1-10 are the hidden factors from Marchenko_PC.

`1. Number of covariates in the raw covariates file: 4. `    

- msex: This refers to the genetic sex of the individual. In many genetic studies, sex is an important covariate because males and females can have different baseline levels of gene expression and can respond differently to genetic and environmental factors.

- age_death: This refers to the age at which the individual died. Age can influence gene expression patterns, with some genes being more or less active at different stages of life. Including age at death as a covariate can help account for age-related variations in gene expression.

- pmi: This stands for "post-mortem interval." It refers to the time elapsed between an individual's death and when their tissue or samples were collected or preserved. PMI can influence the quality and stability of RNA and other molecules in the sample. By including PMI as a covariate, the analysis can account for potential biases or noise introduced by variations in sample quality due to differing post-mortem intervals.

`2. Number of covariates from genotype pca: 15. `    
Including PCs from genotype data as covariates helps in controlling for potential confounders, especially population stratification, thereby reducing the risk of false-positive findings and increasing the robustness of the eQTL analysis. 15 PC will explain 80% variation in the data, so we include 15 PC in this case.

`3. Number of covariates from hidden factor: 10. `    
Hidden factors represent sources of variability in the data that are not directly measured but can significantly influence the observed variables. In the context of genomics, these hidden factors can arise from various sources, including batch effects, technical artifacts, or other unmeasured biological factors that can influence gene expression levels. Identifying and accounting for these hidden factors is crucial to reduce confounding and enhance Interpretability.


In summary, including these covariates in the analysis helps to ensure that the observed associations between genotypes and gene expression are not confounded by these other factors.

## Output Files

| File | Description |
| --- | --- |
| `output/covariate/protocol_example.covariates.protocol_example.genotype...prune.pca.gz` | Merged covariates: known covariates + genotype PCs (output of Step 1). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.residual.bed.gz` | Phenotype residuals after regressing out known covariates (intermediate, Step 2). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.Marchenko_PC.gz` | **Final covariate file**: known covariates + genotype PCs + hidden factors, ready for QTL analysis. |


## Anticipated Results

The final `*.Marchenko_PC.gz` file contains, as rows, the 4 known covariates (`msex`, `age_death`, `pmi`, `study`), 15 genotype PCs (`PC1`-`PC15`), and the hidden factors (`Hidden_Factor_PC1` onward) inferred by Marchenko-Pastur, with one column per sample. This single file is the covariate input for the downstream TensorQTL association analysis.
